## Module 4 — Core Concepts of LangGraph



### 🔁 Quick Recap — What is LangGraph?

> 📌 *LangGraph is an orchestration framework for building intelligent, stateful, and multi-step LLM workflows.*

When you give LangGraph a workflow to execute, it represents that workflow as a **graph** — where:
- Every **node** = one task in the workflow (calling an LLM, calling a tool, making a decision)
- Every **edge** = the connection between tasks (defining what runs next)

Once the graph is built, you provide input to the first node and trigger execution. LangGraph automatically executes all nodes in the correct order until the workflow completes.

Beyond basic graph execution, LangGraph also gives you:

| Feature | What it enables |
|---|---|
| ⚡ **Parallel execution** | Multiple nodes run simultaneously |
| 🔁 **Loops** | Execution can cycle back to previous nodes |
| 🔀 **Branching** | Conditional routing to different nodes |
| 💾 **Memory** | Conversations and task history are recorded |
| ▶️ **Resumability** | If a workflow breaks mid-run, resume from that exact point |

> 💡 *These features combined make LangGraph the ideal framework for building agentic and production-grade AI applications.*

---

### 🧩 Core Concept 1 — LLM Workflows

Before diving into LangGraph's internals, let's make sure the term **LLM Workflow** is crystal clear — because LangGraph is fundamentally a framework for building them.

**What is a Workflow?**

> 📌 *A workflow is a series of tasks executed in a specific order to achieve a goal.*

Our automated hiring example was a workflow — to hire a Backend Engineer, we executed a sequence of steps: create JD → post JD → monitor applications → shortlist → interview → offer → onboard. Each step had to happen in the right order for the overall goal to be achieved.

**What is an LLM Workflow?**

An LLM workflow is a workflow where **multiple tasks depend on or use LLMs** during execution.

In the hiring example:
- Writing the JD → needed an LLM
- Scoring resumes → needed an LLM
- Generating interview questions → needed an LLM

Any workflow that uses LLMs at one or more steps is an **LLM workflow**. And LangGraph is the framework designed to build, orchestrate, and execute them.

**Key properties of LLM Workflows:**
- Each step performs a distinct task — prompting, reasoning, tool calling, memory access, or decision making
- They can be linear, parallel, branching, or looped
- They enable complex behaviors like retries, multi-agent coordination, and tool-augmented reasoning

> 💡 Every application has its own unique workflow. An automated hiring system, a call center automation, a customer support agent — each has a different flow. But certain *patterns* of workflows appear repeatedly across different applications. We'll study those common patterns next.

---



---

### 🔄 Common LLM Workflow Patterns

Every application has a unique workflow, but certain *patterns* appear repeatedly across different systems. Understanding these patterns helps you recognize them in the wild and know exactly how to implement them. Let's go through them one by one.

---

#### Pattern 1 — Prompt Chaining 🔗

Prompt chaining is one of the most common LLM workflow patterns you'll encounter.

> 📌 *Prompt chaining means calling an LLM multiple times in sequence — where the output of one LLM call becomes the input for the next.*

**Why use it?**  
When you have a complex task, breaking it into smaller sub-tasks and solving them step by step often produces much better results than trying to solve everything in a single LLM call.

**Example — Report Generator:**  
A user gives you a topic. You need to generate a detailed report on it.

Instead of asking the LLM to directly produce the final report, you break it down:
```
Topic Input
    ↓
LLM Call 1: "Generate an outline for this topic"
    ↓
Outline Output
    ↓
[Optional Check: Is the outline well-structured?]
    ↓
LLM Call 2: "Write a detailed report based on this outline"
    ↓
Final Report
```

You can also insert **gates** between steps — programmatic checks that validate the intermediate output before passing it forward. For example: *"If the report exceeds 5000 words, exit and flag it."* This keeps the workflow on track without needing human intervention at every step.

> 💡 *Use prompt chaining when you want to decompose a complex task into sequential sub-tasks, with optional quality checks between steps.*

---

#### Pattern 2 — Routing 🔀

Routing is the second fundamental pattern — and one you'll use constantly in real-world systems.

> 📌 *Routing means using an LLM to classify an incoming request and then directing it to the most appropriate handler.*

**Example — Customer Support Chatbot:**  
A customer support bot receives queries of all kinds — refund requests, technical issues, sales inquiries. Rather than using one generic LLM to handle everything (which produces mediocre results), you:
```
Customer Query
    ↓
Router LLM: "What type of query is this?"
    ↓
├── Refund query    → Refund Specialist LLM
├── Technical query → Technical Support LLM
└── Sales query     → Sales LLM
```

The first LLM acts as a **decision-maker** — it doesn't answer the query itself, it just classifies it and routes it to the right specialist. Each specialist LLM is optimized for its own type of query, producing much better outcomes than a one-size-fits-all approach.

> 💡 *Use routing when different types of inputs require different handling, and you want to separate concerns by routing to specialized handlers.*

---


---

#### Pattern 3 — Parallelization ⚡

> 📌 *Parallelization means breaking a task into multiple independent sub-tasks, executing them simultaneously, and then aggregating all results into a final outcome.*

This pattern is about speed and efficiency. When sub-tasks don't depend on each other's results, there's no reason to run them sequentially — run them all at once and merge the results at the end.

**Example — Content Moderation for YouTube:**

When a video is uploaded to YouTube, it needs to be checked across multiple dimensions before going live. These checks are independent of each other — you don't need to know the community guidelines result before you can check for misinformation. So they can all run in parallel.
```
Video Uploaded
    ↓
Transcription Generated
    ↓
┌─────────────────────────────────────┐
│  LLM 1          LLM 2         LLM 3 │
│  Community      Misinformation Sexual│
│  Guidelines     Check         Content│
│  Check                        Check │
└─────────────────────────────────────┘
    ↓         ↓            ↓
         Aggregator
    (combines all three results)
    ↓
Decision: Publish ✅ or Flag 🚩
```

All three LLM calls run simultaneously. The aggregator collects their outputs and makes the final publish/flag decision based on all three results together.

**Why is this powerful?**
- If each check takes 5 seconds sequentially → 15 seconds total
- Running in parallel → ~5 seconds total
- Same quality, 3x faster ⚡

> 💡 *Use parallelization when a task can be broken into independent sub-tasks — tasks that don't need each other's results to run. Execute them simultaneously and aggregate at the end.*

---



---

#### Pattern 4 — Orchestrator-Worker 🎯

> 📌 *The Orchestrator-Worker pattern is similar to parallelization — a task is split into sub-tasks that run in parallel and results are aggregated. The key difference: the sub-tasks are not predefined. They are determined dynamically at runtime by an Orchestrator LLM based on the input.*

**Parallelization vs Orchestrator-Worker — The Key Distinction:**

| | Parallelization | Orchestrator-Worker |
|---|---|---|
| **Sub-tasks defined** | Upfront by developer | Dynamically by Orchestrator LLM |
| **Nature of sub-tasks** | Fixed and known | Varies based on input |
| **Example** | Always check: guidelines + misinformation + sexual content | What to check depends on what the input is |

---

**Example — Research Assistant:**

You're building a research assistant that takes any query and produces a detailed research report. The challenge is that *how* you research depends entirely on *what* you're researching.
```
User Query: "What is CRISPR gene editing?"
    ↓
Orchestrator LLM analyzes the query
"This is a scientific topic → search academic sources"
    ↓
┌────────────────────────────────────┐
│ Worker 1          Worker 2          │
│ Google Scholar    PubMed            │
│ search            search            │
└────────────────────────────────────┘
         ↓               ↓
              Aggregator
         Combines findings into report
```

But if the query changes:
```
User Query: "2024 US election results"
    ↓
Orchestrator LLM analyzes the query
"This is a political/news topic → search news sources"
    ↓
┌────────────────────────────────────┐
│ Worker 1          Worker 2          │
│ Google News       Reuters           │
│ search            search            │
└────────────────────────────────────┘
         ↓               ↓
              Aggregator
         Combines findings into report
```

The Orchestrator LLM acts like a smart manager — it reads the input, decides what kind of work needs to be done, assigns the right tasks to the right workers, and then an aggregator combines all results into the final output.

> 💡 *Use the Orchestrator-Worker pattern when the nature of your sub-tasks can't be predetermined — they need to be intelligently decided at runtime based on what the input actually is.*

---



---

#### Pattern 5 — Evaluator-Optimizer 🔄✨

> 📌 *The Evaluator-Optimizer pattern is used when a task can't be solved perfectly in one shot — it requires iterative refinement. A Generator LLM produces a solution, an Evaluator LLM critiques it, and the loop repeats until the output meets the quality bar.*

**The Core Insight — Great Work Takes Iteration**

Think about how a poet or novelist writes. They don't produce a masterpiece in one sitting. They write a first draft, identify what's missing, incorporate that feedback into a second draft, and keep iterating until the result feels right. The same principle applies here.

Tasks like writing emails, blogs, poems, or stories have no single "correct" answer — quality is subjective and improves with iteration. The Evaluator-Optimizer pattern automates this refinement loop.

**How it works:**
```
Task Input: "Write a blog post about LangGraph"
    ↓
Generator LLM
Produces first draft (Solution v1)
    ↓
Evaluator LLM
Checks against evaluation criteria
    ↓
┌─────────────────────────────┐
│  Accepted? ✅               │
│  → Return final output      │
│                             │
│  Rejected? ❌               │
│  → Generate feedback        │
│  → Send back to Generator   │
└─────────────────────────────┘
    ↓ (if rejected)
Generator LLM
Produces improved draft using feedback (Solution v2)
    ↓
Evaluator LLM
Re-evaluates...
    ↓
[Loop continues until Evaluator is satisfied]
    ↓
Final Output ✅
```

The **Generator** focuses purely on producing content. The **Evaluator** focuses purely on quality assessment — it has clear evaluation criteria (tone, length, accuracy, structure, etc.) and either accepts the solution or sends back specific, actionable feedback. The generator uses that feedback to produce a better version, and the loop runs until the evaluator is satisfied.

> 💡 *Use Evaluator-Optimizer for creative or open-ended tasks where quality improves with iteration — writing, code generation, summarization, translation, or any task where "good enough" isn't good enough.*

---

### 🗺️ Summary — The 5 Common LLM Workflow Patterns

| Pattern | Core Idea | Best For |
|---|---|---|
| **Prompt Chaining** | Sequential LLM calls, output of one feeds into next | Complex tasks broken into ordered steps |
| **Routing** | LLM classifies input and directs it to the right handler | Different input types needing different specialists |
| **Parallelization** | Independent sub-tasks run simultaneously, results aggregated | Fixed parallel checks where sub-tasks are predefined |
| **Orchestrator-Worker** | Orchestrator LLM dynamically assigns sub-tasks to workers | Tasks where the nature of sub-tasks varies by input |
| **Evaluator-Optimizer** | Generator + Evaluator loop until quality bar is met | Creative or open-ended tasks requiring iteration |

> 📌 *All five of these patterns will be built hands-on in this series. Recognizing which pattern fits a given problem is a core skill for any agentic AI developer.*

---



---

### 🧩 Core Concept 2 — Graphs, Nodes, and Edges

> 📌 *This is the single most important core concept in LangGraph. Everything else builds on top of it.*

#### The Big Idea

LangGraph represents any LLM workflow as a **graph**. To understand what that means in practice, let's walk through a real example.

---

#### Example — UPSC Essay Practice Platform 📝

Imagine building a website where UPSC aspirants can practice writing essays (a high-weightage component of the UPSC Mains exam). Here's how the platform works:

1. The system generates an essay topic for the user
2. The user writes and submits their essay
3. The system evaluates the essay across 3 dimensions:
   - **Clarity of thought** (0–5)
   - **Depth of analysis** (0–5)
   - **Language & vocabulary** (0–5)
4. Total score out of 15. Threshold = 10.
   - Score ≥ 10 → Congratulate the user ✅
   - Score < 10 → Provide detailed feedback ❌
5. Ask the user: "Do you want to revise and resubmit?"
   - Yes → Loop back to essay submission
   - No → End

This is an LLM workflow. Now — how does LangGraph represent it?

---

#### Step 1 — Convert the Goal into Actionable Steps

Before writing any code, translate the high-level goal into discrete steps:
```
1. Generate Topic
2. Collect Essay (from user)
3. Evaluate Essay (3 dimensions in parallel)
4. Aggregate Scores
5. Final Evaluation (pass/fail decision)
6. Provide Feedback (if failed)
7. Ask to Revise → loop back if yes, end if no
```

#### Step 2 — Represent as a Graph

Each step becomes a **Node**. The connections between steps become **Edges**.
```
[Generate Topic]
      ↓
[Collect Essay]
      ↓
┌──────────────────────────────────┐
│ Evaluate       Evaluate     Evaluate │
│ Clarity        Depth        Language  │
└──────────────────────────────────┘
      ↓
[Aggregate Scores]
      ↓
[Final Evaluation]
      ↓
   ┌──────────────┐
   ✅ Pass        ❌ Fail
   End            ↓
              [Give Feedback]
                  ↓
            Revise? (Yes/No)
            ↓           ↓
      [Back to        End
      Collect Essay]
```

---

#### What Are Nodes?

> 📌 *A Node represents a single task in the workflow.*

Behind the scenes, every node in LangGraph is simply a **Python function**. Nothing more. If you can write a Python function, you can create a node.
```python
def generate_topic(state):
    # LLM generates a topic
    ...
    return updated_state

def evaluate_clarity(state):
    # LLM evaluates clarity of thought
    ...
    return updated_state
```

#### What Are Edges?

> 📌 *Edges define the flow of execution — which node runs after which.*

Edges answer the question: *"After this node completes, where does execution go next?"*

LangGraph supports four types of edges:

| Edge Type | What it does | Example |
|---|---|---|
| **Sequential** | One node leads directly to the next | Generate Topic → Collect Essay |
| **Parallel** | One node triggers multiple nodes simultaneously | Evaluate all 3 dimensions at once |
| **Conditional** | Based on a condition, flow goes one way or another | Score ≥ 10 → Congratulate, else → Feedback |
| **Loop** | Flow cycles back to a previous node | Revise → back to Collect Essay |

---

#### The Key Insight

> 📌 **Nodes tell LangGraph *what* to do. Edges tell LangGraph *when* to do it.**

Because the underlying data structure is a graph — which is inherently non-linear — LangGraph can naturally express sequential flows, parallel execution, branching, and looping. All within a clean, unified structure. No glue code required.

A LangGraph workflow is essentially: *a set of Python functions, interconnected by edges that define the execution order.*

> 💡 *This will feel very concrete once we start writing code. The conceptual picture is: every task is a node, every transition is an edge, and the graph is your complete workflow.*

---



---

### 🧩 Core Concept 3 — State

> 📌 *In LangGraph, State is a shared memory that flows through your entire workflow. It holds all the data being passed between nodes as your graph runs.*

---

#### What is State?

Every LLM workflow needs certain pieces of data to guide its execution from start to finish. This data has two key properties:

1. **Required throughout execution** — multiple nodes need access to it
2. **Evolves over time** — its values change as the workflow progresses

This collection of data points is called **State**.

**Back to our UPSC Essay example:**

Think about what data is needed as the workflow runs:
```python
class EssayState(TypedDict):
    essay_topic: str          # Generated at start, used later for context
    essay_text: str           # Submitted by user, re-written if they revise
    clarity_score: float      # Set by Evaluate Clarity node (0-5)
    depth_score: float        # Set by Evaluate Depth node (0-5)
    language_score: float     # Set by Evaluate Language node (0-5)
    overall_score: float      # Computed by Aggregate node
    feedback: str             # Generated if score < threshold
    revision_requested: bool  # User's choice to revise or not
```

Each of these fields starts empty or `None` and gets populated as nodes execute. When the user revises their essay, `essay_text` gets overwritten with the new version, scores reset, and the whole evaluation loop runs again. The state evolves continuously as execution progresses.

---

#### How State Flows Through the Graph

The most powerful thing about State in LangGraph: **every node has access to the full State at all times.**

The execution flow works like this:
```
Graph starts with initial State (mostly empty)
    ↓
Node 1 (Generate Topic) receives State
→ adds essay_topic to State
→ passes updated State forward
    ↓
Node 2 (Collect Essay) receives State
→ adds essay_text to State
→ passes updated State forward
    ↓
Node 3a (Evaluate Clarity) receives State
→ reads essay_text, sets clarity_score
→ passes updated State forward
    ↓
... and so on through every node
```

Every node gets the **full current State as input**, makes its changes, and passes the **updated State** as output to the next node. No node ever needs to worry about what happened before it — all that context is already in the State.

Two critical properties of State:
- ✅ **Shared** — accessible by every node in the graph
- ✅ **Mutable** — any node can read from and write to it

---

#### How to Define State in Code

State is implemented as a **TypedDictionary** in Python — a special class where you define all your fields with their types upfront:
```python
from typing import TypedDict

class EssayState(TypedDict):
    essay_topic: str
    essay_text: str
    clarity_score: float
    depth_score: float
    language_score: float
    overall_score: float
    feedback: str
    revision_requested: bool
```

You can also use a **Pydantic model** if you want built-in validation. But TypedDict is the most common approach in LangGraph.

Once defined, LangGraph automatically handles passing this State object to every node, and collecting each node's updates back into the shared State.

---

> 📌 **The mental model:** Think of State as a *shared whiteboard* that every node in the graph can read from and write to. As execution moves from node to node, each one picks up the whiteboard, does its work, updates what it needs to, and passes the whiteboard forward.

| Property | Description |
|---|---|
| **Shared** | All nodes access the same State object |
| **Mutable** | Any node can update any field |
| **Evolving** | Values change as execution progresses |
| **Implementation** | TypedDict or Pydantic model |

---



---

### 🧩 Core Concept 4 — Reducers

> 📌 *Reducers define **how** updates from nodes are applied to the shared State. They answer the question: when a node writes to a State field, should the new value **replace** the old one, **append** to it, or **merge** with it?*

This concept is closely tied to State — so let's build on what we already know.

---

#### The Default Behavior — Replace

By default, when a node updates a State field, it **replaces** the previous value. This makes perfect sense for most fields.

**Example — Simple Math Workflow:**
```python
class MathState(TypedDict):
    num1: int
    num2: int
    result: int
```
```
Node 1 (Get Input):   result = None
Node 2 (Sum):         result = 5 + 6 = 11   ← replaces None
Node 3 (Multiply):    result = 11 * 2 = 22  ← replaces 11
```

Each node overwrites `result` with the new value. The old value is gone. This is correct here — we only ever need the latest computed value.

---

#### When Replace Breaks Things — The Chatbot Problem

Now consider a simple chatbot workflow:
```python
class ChatState(TypedDict):
    messages: str   # stores the current message
```
```
Human Node:   messages = "Hi, my name is Gourab"
LLM Node:     messages = "Hi! How can I help you?"  ← replaces!
Human Node:   messages = "Can you tell me my name?" ← replaces!
LLM Node:     ??? — the name was never seen, it was erased!
```

The LLM has no idea what the user's name is, because every new message replaced the previous one. The entire conversation history is lost after each step.

**What we actually need here is Append — not Replace.**
```
messages = ["Hi, my name is Gourab"]
messages = ["Hi, my name is Gourab", "Hi! How can I help you?"]
messages = ["Hi, my name is Gourab", "Hi! How can I help you?", "Can you tell me my name?"]
```

Now the LLM receives the full conversation history at every step and can answer correctly.

---

#### Another Example — The UPSC Essay Evolution

Back to our essay platform. Say a student fails, revises, fails again, revises again. With the default **Replace** reducer:
```
Attempt 1: essay_text = "First draft..."
Attempt 2: essay_text = "Second draft..."  ← first draft lost!
Attempt 3: essay_text = "Third draft..."   ← second draft lost!
```

But what if the student wants to see their own progress — how their writing improved across attempts? In that case you'd want **Append**:
```
essay_texts = ["First draft...", "Second draft...", "Third draft..."]
```

All three are preserved, and the student can review their evolution.

---

#### The Three Reducer Strategies

| Strategy | Behavior | Use when |
|---|---|---|
| **Replace** (default) | New value overwrites old | Only latest value matters (scores, flags, status) |
| **Append** | New value is added to existing list | History matters (chat messages, essay drafts, logs) |
| **Merge** | New dict/object is merged with existing | Combining partial results (e.g., parallel node outputs) |

In code, you attach a reducer to a specific State field using Python's `Annotated` type:
```python
from typing import Annotated
from operator import add

class ChatState(TypedDict):
    # Default replace behavior
    topic: str
    
    # Append reducer — new messages add to list, not replace it
    messages: Annotated[list, add]
```

Each key in State can have its own reducer. One field can use Replace while another uses Append — whatever makes sense for that piece of data.

> 📌 *Reducers are especially important in **parallel workflows** — when multiple nodes write to the same State field simultaneously, you need a clear rule for how those writes get combined. We'll see this hands-on when we build the parallelization workflow.*

---



---

### 🧩 Core Concept 5 — LangGraph's Execution Model

> 💡 *Interesting fact: LangGraph's execution model is inspired by **Google Pregel** — Google's system for large-scale graph processing used across many of their products.*

Understanding how LangGraph executes a workflow under the hood will make debugging and building much more intuitive.

---

#### The Three Phases of Execution

**Phase 1 — Graph Definition**

Before any execution, you define three things:
- **Nodes** — the Python functions (tasks)
- **Edges** — the connections between nodes
- **State** — the TypedDict that flows through the graph

**Phase 2 — Compilation**

You call `.compile()` on the graph. This step validates the graph's structure — catching issues like orphaned nodes (nodes that aren't connected to anything) or structural inconsistencies before runtime.
```python
app = graph.compile()
```

**Phase 3 — Execution (Invocation)**

You invoke the graph by passing the initial State to the first node:
```python
app.invoke({"essay_topic": None, "essay_text": None, ...})
```

From here, LangGraph takes over completely.

---

#### How Execution Flows — Message Passing

Once invoked, here's what happens automatically under the hood:
```
Initial State passed to Node 1
    ↓
Node 1 activates → its Python function runs
→ makes partial updates to State
→ updated State sent forward via Edge  ← this is Message Passing
    ↓
Node 2 activates → its Python function runs
→ makes partial updates to State
→ updated State sent forward via Edge
    ↓
... continues until no active nodes remain
    ↓
Workflow terminates ✅
```

> 📌 **Message Passing** = sending the updated State object from one node to the next via edges. You never manually call nodes in sequence — LangGraph handles the entire chain automatically.

---

#### Supersteps — Why Not Just "Steps"?

In a simple sequential graph, each round of execution involves exactly one node at a time. But what about parallel nodes?
```
Node A completes
    ↓ ↓ ↓
Node B   Node C   Node D  ← all three activate simultaneously
```

Here, one "round" of execution involves three parallel steps happening at once. Calling this a single "step" would be misleading — so LangGraph calls it a **Superstep**.

> 📌 **Superstep** = one round of execution in LangGraph. A superstep can contain one step (sequential) or multiple steps (parallel). When parallel nodes all complete their work, their State updates are merged via Reducers before being passed forward.

---

#### When Does Execution Stop?

LangGraph terminates the workflow when **both** of these conditions are true:
- No nodes are currently active
- No messages are being passed through any edges
```
All nodes idle + No edges carrying messages → Workflow complete ✅
```

---

#### Complete Execution Flow Summary
```
1. Graph Definition  → Define nodes, edges, State
2. Compilation       → Validate graph structure
3. Invocation        → Pass initial State to first node
4. Message Passing   → State flows node → edge → node automatically
5. Supersteps        → Sequential or parallel rounds of execution
6. Termination       → No active nodes, no messages in transit
```

> 📌 *You only ever trigger the first node. Everything after that — sequencing, parallelism, branching, looping, state updates — LangGraph manages entirely on its own.*

---

### 🗺️ Module 4 — Complete Summary of Core Concepts

| Concept | What it is |
|---|---|
| **LLM Workflow** | Series of tasks using LLMs to achieve a goal |
| **Workflow Patterns** | Prompt Chaining, Routing, Parallelization, Orchestrator-Worker, Evaluator-Optimizer |
| **Graph** | The representation of a workflow in LangGraph |
| **Node** | A single task — implemented as a Python function |
| **Edge** | Connection between nodes — defines execution flow (sequential, parallel, conditional, loop) |
| **State** | Shared TypedDict accessible and mutable by all nodes |
| **Reducer** | Rule for how State updates are applied — replace, append, or merge |
| **Message Passing** | The mechanism of sending updated State through edges |
| **Superstep** | One round of execution — can include multiple parallel steps |

---

